# Uni-D 배터리 열폭주 6단계 분류 — 베이스라인 코드

각 관측 시점의 **열화상 이미지 + 센서값**으로 `label`(1~6)을 예측하는 출발점입니다.

| 구성 | 내용 |
|---|---|
| 센서 | `meas_v`, `meas_a`, `meas_t1`, `meas_t2`, `meas_p` |
| 이미지 | 열화상 밝기 통계 6개 (평균·표준편차·최댓값·95/99 백분위·고온 픽셀 비율) |
| 시계열 | 같은 실험 안 **과거 30초**의 평균·최댓값·변화량 (`relative_time_s` 기준) |
| 모델 | 로지스틱 회귀 (클래스 가중치 유무를 검증으로 선택) |

흐름: 데이터 준비 → 피처 생성 → 실험 단위 교차검증·Validation 평가 → **규정 준수 자가 점검(B·C·D)** → Test 추론 → `submission.csv`

- CPU만으로 실행됩니다. 대회 제공 GPU 서버(Jupyter)와 Colab 모두에서 동작합니다.
- 이 노트북은 데이터 로딩과 결합 방법을 보여주는 **최소 baseline**입니다. 이미지는 단순 통계만 사용하므로 CNN 등 이미지 모델링, 피처 설계, 모델 선택에서 개선할 부분들이 많습니다.
- 최종 제출용 `model.py` / `train.py` / `test.py`는 이 노트북과 같은 코드로 함께 제공됩니다(7번 참고).

## 0. 설정
`DATA_DIR`(압축을 푼 `battery_data_light` 폴더) 또는 `DATA_TAR`(참가자용 TAR) 중 하나만 맞게 지정해주세요.

In [ ]:
from pathlib import Path
import tarfile
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupKFold

DATA_DIR = None                                             # 예: Path('/home/user/data/battery_data_light')
DATA_TAR = Path('./battery_data_for_participants.tar')      # DATA_DIR가 None이면 이 TAR를 WORK_DIR에 풉니다.
WORK_DIR = Path('./work')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'

# Colab에서 Drive의 TAR를 쓰는 경우에만 아래 두 줄의 주석을 해제하십시오.
# from google.colab import drive; drive.mount('/content/drive')
# DATA_TAR = Path('/content/drive/MyDrive/battery_data_for_participants.tar')

## 1. 데이터 준비 및 확인

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
if DATA_DIR is None:
    DATA_DIR = WORK_DIR / 'data' / 'battery_data_light'
    if not DATA_DIR.is_dir():
        assert DATA_TAR.is_file(), f'TAR 경로를 확인하십시오: {DATA_TAR}'
        print('TAR 압축 해제 중... (처음 한 번만)')
        with tarfile.open(DATA_TAR, 'r:*') as archive:
            try:
                archive.extractall(WORK_DIR / 'data', filter='data')
            except TypeError:  # Python 3.11 이하
                archive.extractall(WORK_DIR / 'data')
DATA_DIR = Path(DATA_DIR)
assert DATA_DIR.is_dir(), f'데이터 폴더가 없습니다: {DATA_DIR}'

def read_split(split):
    return pd.read_csv(DATA_DIR / split / f'{split}.csv', dtype={'sample_id': 'string', 'experiment_id': 'string'})

frames = {split: read_split(split) for split in ['train', 'validation', 'test']}
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'sample_id': 'string'})
train, validation, test = frames['train'], frames['validation'], frames['test']

for split, frame in frames.items():
    assert frame.sample_id.notna().all() and frame.sample_id.is_unique, split
    assert frame.experiment_id.notna().all(), split
    print(f'{split}: {len(frame):,}개 / {frame.experiment_id.nunique()}개 실험')
for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(frames[left].experiment_id).isdisjoint(frames[right].experiment_id), '실험 분할 중복'
assert 'label' not in test.columns
assert sample_submission.columns.tolist() == ['sample_id', 'label']
assert set(sample_submission.sample_id) == set(test.sample_id), 'Test와 제출 양식 ID 불일치'
display(train.head())

관측 간격은 실험마다 다를 수 있습니다. 특히 Test 일부 실험은 Train·Validation보다 간격이 넓으므로, 시계열 피처는 **행 수가 아니라 `relative_time_s`(초)** 기준으로 만들어야 합니다.

In [ ]:
def median_gap(frame):
    gaps = frame.sort_values(['experiment_id', 'relative_time_s']).groupby('experiment_id').relative_time_s.diff()
    return gaps.groupby(frame.experiment_id).median()

display(pd.DataFrame({split: median_gap(f).describe() for split, f in frames.items()}).round(2))
display(pd.DataFrame({'train': train.label.value_counts(), 'validation': validation.label.value_counts()}).sort_index())

## 2. 피처 정의
아래 코드는 제출 템플릿의 `model.py`와 같습니다.

**시계열 피처 규정 준수 방식**
- 같은 `experiment_id` 안에서 `relative_time_s`가 현재 이하인 관측만 사용합니다.
- 실험 전체 평균·최댓값·길이·순위 같은 값은 쓰지 않습니다.
- 실험마다 따로 계산하므로, 입력에 다른 실험이나 미래 관측이 있든 없든 결과가 같습니다(5번에서 확인).
- 결측 대체·표준화 기준은 학습 데이터에서만 계산합니다(파이프라인 안에서 처리).

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from PIL import Image
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
LABELS = list(range(1, 7))
SENSORS = ['meas_v', 'meas_a', 'meas_t1', 'meas_t2', 'meas_p']
IMAGE_STATS = ['img_mean', 'img_std', 'img_max', 'img_p95', 'img_p99', 'img_hot_ratio']
# 시계열 피처를 만들 신호: 센서 5개 + 이미지 고온 통계 2개
WINDOW_SIGNALS = SENSORS + ['img_max', 'img_p99']
WINDOW_S = 30.0          # 과거 윈도우 길이(초). 행 수가 아니라 relative_time_s 기준입니다.
HOT_THRESHOLD = 0.8      # 밝기 0~1 기준 고온 픽셀 임계값(고정값, 데이터로 학습하지 않음)


# ---------------------------------------------------------------------------
# 1. 이미지 경로 연결
# ---------------------------------------------------------------------------
def resolve_image_paths(frame: pd.DataFrame, roots: list[Path]) -> pd.Series:
    """CSV의 image_path를 실제 파일 경로로 바꿉니다.

    roots를 순서대로 기준 폴더로 시도하고, 모두 실패하면 파일명으로 찾습니다.
    """
    rel = frame['image_path'].astype(str)
    for root in roots:
        candidate = rel.map(lambda p: Path(root) / p)
        if candidate.map(Path.is_file).all():
            return candidate
    # 경로 규칙이 맞지 않으면 파일명 기준으로 검색합니다.
    index = {}
    for root in roots:
        for path in Path(root).rglob('*'):
            if path.is_file():
                index.setdefault(path.name, path)
    candidate = rel.map(lambda p: index.get(Path(p).name))
    missing = candidate.isna().sum()
    assert missing == 0, f'이미지 {missing}개를 찾지 못했습니다. image_root를 확인하십시오. 예: {rel[candidate.isna()].iloc[0]}'
    return candidate


# ---------------------------------------------------------------------------
# 2. 이미지 통계 피처 (샘플마다 독립 계산 → 다른 행 정보 사용 없음)
# ---------------------------------------------------------------------------
def _image_stats(path) -> list[float]:
    with Image.open(path) as img:
        arr = np.asarray(img.convert('L'), dtype=np.float32) / 255.0
    return [
        float(arr.mean()),
        float(arr.std()),
        float(arr.max()),
        float(np.percentile(arr, 95)),
        float(np.percentile(arr, 99)),
        float((arr >= HOT_THRESHOLD).mean()),
    ]


def extract_image_features(frame: pd.DataFrame, paths: pd.Series, n_jobs: int = -1) -> pd.DataFrame:
    """sample_id를 인덱스로 하는 이미지 통계 표를 만듭니다."""
    rows = Parallel(n_jobs=n_jobs, batch_size=64)(delayed(_image_stats)(p) for p in paths)
    return pd.DataFrame(rows, columns=IMAGE_STATS, index=pd.Index(frame['sample_id'].values, name='sample_id'))


# ---------------------------------------------------------------------------
# 3. 과거 방향 시계열 피처
#    같은 experiment_id 안에서 relative_time_s <= 현재 시점인 관측만 사용합니다.
#    실험 전체 통계(평균·최댓값·길이·순위 등)는 사용하지 않습니다.
# ---------------------------------------------------------------------------
def _causal_one_experiment(t: np.ndarray, x: np.ndarray, window_s: float):
    """한 실험(시간순 정렬)에 대해 행마다 (t-window, t] 평균·최댓값과 window 이전 값과의 차이를 계산합니다."""
    n = len(t)
    mean = np.full(n, np.nan)
    vmax = np.full(n, np.nan)
    delta = np.full(n, np.nan)
    start = np.searchsorted(t, t - window_s, side='right')   # 윈도우 시작 위치
    end = np.searchsorted(t, t, side='right')                 # 같은 시각 관측까지 포함(<= t)
    lag = np.searchsorted(t, t - window_s, side='right') - 1  # t-window 이하의 마지막 관측
    for i in range(n):
        w = x[start[i]:end[i]]
        w = w[~np.isnan(w)]
        if len(w):
            mean[i] = w.mean()
            vmax[i] = w.max()
        if lag[i] >= 0:
            delta[i] = x[i] - x[lag[i]]
    return mean, vmax, delta


def add_causal_features(base: pd.DataFrame, signals: list[str], window_s: float = WINDOW_S) -> pd.DataFrame:
    """실험마다 따로 계산하므로 다른 실험·미래 관측이 입력에 있든 없든 결과가 같습니다."""
    w = int(window_s)
    cols = [f'{s}_{k}{w}s' for s in signals for k in ('mean', 'max', 'delta')]
    out = pd.DataFrame(np.nan, index=base.index, columns=cols)
    times = base['relative_time_s'].to_numpy(dtype=float)
    for _, idx in base.groupby('experiment_id', sort=False).indices.items():
        idx = idx[np.argsort(times[idx], kind='mergesort')]
        t = times[idx]
        for s in signals:
            m, mx, d = _causal_one_experiment(t, base[s].to_numpy(dtype=float)[idx], window_s)
            out.iloc[idx, out.columns.get_loc(f'{s}_mean{w}s')] = m
            out.iloc[idx, out.columns.get_loc(f'{s}_max{w}s')] = mx
            out.iloc[idx, out.columns.get_loc(f'{s}_delta{w}s')] = d
    return out


# ---------------------------------------------------------------------------
# 4. 전체 피처 조립
# ---------------------------------------------------------------------------
def numeric(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    return frame[cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)


def build_features(frame: pd.DataFrame, image_features: pd.DataFrame) -> pd.DataFrame:
    """frame에 들어 있는 행만으로 피처를 만듭니다. 다른 split·다른 실험의 행은 쓰지 않습니다."""
    base = frame[['sample_id', 'experiment_id']].copy()
    base['relative_time_s'] = pd.to_numeric(frame['relative_time_s'], errors='coerce')
    base[SENSORS] = numeric(frame, SENSORS)
    img = image_features.reindex(frame['sample_id'].values)
    assert img.notna().all(axis=1).all(), '이미지 피처가 없는 샘플이 있습니다.'
    base[IMAGE_STATS] = img.values
    causal = add_causal_features(base, WINDOW_SIGNALS)
    return pd.concat([base[['relative_time_s'] + SENSORS + IMAGE_STATS], causal], axis=1)


def build_model(class_weight=None) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=1.0, solver='lbfgs', max_iter=3000, class_weight=class_weight, random_state=SEED,
        )),
    ])


## 3. 피처 생성
이미지 통계는 샘플마다 독립적으로 계산하고, 다시 실행할 때를 위해 캐시합니다.

캐시는 `float_precision='round_trip'`으로 읽습니다. 기본 설정으로 읽으면 소수점 끝자리(약 1e-16)가 달라져 학습 결과가 미세하게 바뀌고, `test.py` 결과와 경계 샘플 몇 개가 달라질 수 있습니다(15일 재현 검사 불일치 원인).

In [ ]:
def image_features_for(split, frame):
    cache = WORK_DIR / f'img_features_{split}.csv'
    if cache.is_file():
        img = pd.read_csv(cache, dtype={'sample_id': 'string'},
                          float_precision='round_trip').set_index('sample_id')  # 소수점 손실 없이 읽기
        if set(img.index) == set(frame.sample_id):
            return img
    paths = resolve_image_paths(frame, [DATA_DIR / split, DATA_DIR])
    img = extract_image_features(frame, paths)
    img.to_csv(cache)
    return img

img_feats = {split: image_features_for(split, f) for split, f in frames.items()}
X = {split: build_features(f, img_feats[split]) for split, f in frames.items()}
y_train = train.label.astype(int)
y_validation = validation.label.astype(int)
print('피처 수:', X['train'].shape[1])
display(X['train'].head())

## 4. 모델 학습 및 Validation 평가
평가지표는 **Macro F1**(6개 단계 F1의 단순 평균)이므로 소수 클래스도 중요합니다.

- **Train 실험 단위 교차검증**(`GroupKFold`): 같은 실험이 학습·검증 양쪽에 들어가지 않게 나눕니다. 모델·설정 선택은 이 점수로 하는 것을 권장합니다.
- **Validation**: 학습과 선택에 쓰지 않고 최종 성능 확인에만 사용합니다. Validation 점수를 보고 설정을 고르면, 그 Validation 점수는 실제보다 좋게 나온 값이 됩니다(발표 자료에 성능을 적을 때 주의).

베이스라인은 `class_weight=None`으로 고정합니다. 아래 표는 클래스 가중치의 영향을 보여주는 **참고용 비교**입니다. Train CV와 Validation의 순위가 다를 수 있다는 점도 확인해 보세요(Validation은 실험이 19개뿐이고 5단계 비율도 Train과 다릅니다).

In [ ]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)

n_splits = min(5, train.experiment_id.nunique())
results = []
for name, cw in [('none', None), ('balanced', 'balanced')]:
    cv = []
    for tr_idx, va_idx in GroupKFold(n_splits=n_splits).split(X['train'], y_train, groups=train.experiment_id):
        m = build_model(cw).fit(X['train'].iloc[tr_idx], y_train.iloc[tr_idx])
        cv.append(macro_f1(y_train.iloc[va_idx], m.predict(X['train'].iloc[va_idx])))
    m = build_model(cw).fit(X['train'], y_train)
    results.append({'class_weight': name, 'cv_mean': np.mean(cv), 'cv_std': np.std(cv),
                    'validation_macro_f1': macro_f1(y_validation, m.predict(X['validation']))})
display(pd.DataFrame(results).round(4))

CLASS_WEIGHT = None   # 베이스라인 고정값 (위 표는 참고용)
print('사용 설정: class_weight =', CLASS_WEIGHT)

In [ ]:
model = build_model(CLASS_WEIGHT).fit(X['train'], y_train)
validation_pred = model.predict(X['validation'])
print(f'Validation Accuracy: {accuracy_score(y_validation, validation_pred):.4f}')
print(f'Validation Macro F1: {macro_f1(y_validation, validation_pred):.4f}')
print(classification_report(y_validation, validation_pred, labels=LABELS, digits=4, zero_division=0))
display(pd.DataFrame(confusion_matrix(y_validation, validation_pred, labels=LABELS),
                     index=[f'실제 {l}' for l in LABELS], columns=[f'예측 {l}' for l in LABELS]))

## 5. 규정 준수 자가 점검 (개발 중 빠른 확인)
최종 심사에서 운영진은 `test.py`를 Test 일부만 넣거나 실험을 나누어 다시 실행해 **같은 샘플의 예측이 유지되는지** 확인합니다. 피처나 모델을 바꿀 때마다 이 셀로 먼저 확인하십시오.

이 셀은 매번 **넣은 행만으로 이미지 피처까지 다시 계산**하므로, Test 전체 기준 이미지 정규화나 Test로 fit한 전처리도 여기서 드러납니다.

| 검사 | 방법 | 통과 기준 |
|---|---|---|
| B 미래 잘라내기 | 실험마다 앞쪽 절반만 남김 | 남은 행 예측이 같음 |
| C 실험 분리 | 실험을 두 묶음으로 나눠 따로 실행 | 예측이 같음 |
| D 빈 이미지 | 모든 이미지를 검은 이미지로 바꿈 | 예측이 **바뀌어야** 함 (이미지·센서 모두 사용) |

이 셀은 노트북 안의 코드를 검사합니다. 최종 제출 전에는 실제로 검증받는 `test.py`를 템플릿의 `check_submission.py`로 점검하십시오(재현·반복 실행·행 순서 섞기 포함, GPU 사용 시 수치 오차 안내도 여기서 확인).

In [ ]:
TEST_IMAGE_ROOTS = [DATA_DIR / 'test', DATA_DIR]

def predict_frame(frame):
    """넣은 행만으로 이미지 피처·시계열 피처를 모두 다시 계산해 예측합니다 (test.py와 같은 조건)."""
    paths = resolve_image_paths(frame, TEST_IMAGE_ROOTS)
    img = extract_image_features(frame, paths)
    return pd.Series(model.predict(build_features(frame, img)), index=frame.sample_id.tolist())

def diff_rate(pred):
    return float((pred_A.loc[pred.index] != pred).mean())

pred_A = predict_frame(test)   # 기준 실행 (6번 제출에 사용)
half = test[test.relative_time_s <= test.groupby('experiment_id').relative_time_s.transform('median')]
pred_B = predict_frame(half)
exps = sorted(test.experiment_id.unique())
pred_C = pd.concat([predict_frame(test[test.experiment_id.isin(exps[0::2])]),
                    predict_frame(test[test.experiment_id.isin(exps[1::2])])])

problems = []
for name, pred in [('B 미래 잘라내기', pred_B), ('C 실험 분리', pred_C)]:
    rate = diff_rate(pred)
    print(f'{name}: 불일치 {rate:.2%} ({len(pred)}행)')
    if rate > 0:
        problems.append(name)

# D: 모든 이미지를 같은 크기의 검은 이미지로 바꿔 예측이 달라지는지 확인
first = resolve_image_paths(test.head(1), TEST_IMAGE_ROOTS).iloc[0]
with Image.open(first) as im:
    blank_path = (WORK_DIR / 'blank.png').resolve()
    Image.new(im.mode, im.size).save(blank_path)
d_rate = diff_rate(predict_frame(test.assign(image_path=str(blank_path))))
print(f'D 빈 이미지: 예측 변화 {d_rate:.2%}')
if d_rate == 0:
    problems.append('D 빈 이미지 (이미지를 예측에 쓰지 않음)')

assert not problems, f'자가 점검 실패: {problems} — 미래 관측·다른 실험·Test 전체 기준 정보를 쓰는지, 이미지를 입력으로 쓰는지 확인하십시오.'
print('자가 점검 통과')

## 6. Test 추론 및 제출 파일 생성
제출 양식의 `sample_id` 순서를 유지하고, 라벨은 1~6 정수로 저장합니다.

In [ ]:
submission = sample_submission[['sample_id']].copy()
submission['label'] = submission.sample_id.map(pred_A)
assert submission.label.notna().all()
submission['label'] = submission.label.astype(int)
assert submission.label.isin(LABELS).all() and len(submission) == len(test)

submission.to_csv(SUBMISSION_PATH, index=False)
saved = pd.read_csv(SUBMISSION_PATH, dtype={'sample_id': 'string'})
pd.testing.assert_frame_equal(saved, submission, check_dtype=False)
print('제출 파일 저장:', SUBMISSION_PATH, f'({len(submission)}행)')
display(submission.label.value_counts().sort_index())

## 7. 제출 안내
- 6번에서 만든 `submission.csv`를 DAKER 대회 페이지에 제출하면 리더보드 점수를 확인할 수 있습니다.
- 대회 종료 후 코드 검증에서는 제출한 `test.py`를 다시 실행해 최종 선택 파일과 같은 예측이 나오는지 확인합니다. 노트북 결과와 `test.py` 결과가 다를 수 있으니, **최종 선택할 파일은 `test.py`로 만들어 제출하십시오.**
- 최종 파일을 고르기 전에 템플릿에 함께 들어 있는 `check_submission.py`로 제출 형식, 재현 여부, 규정 준수, 이미지 사용 여부를 확인하십시오.
- 대회가 끝나면 최종 제출 파일을 재현할 수 있는 코드(`model.py`, `train.py`, `test.py` 등)를 따로 제출합니다. 함께 배포한 템플릿에 이 노트북 코드를 스크립트로 옮겨 두었으니 참고하세요.
- 제출 마감 시각과 제출 방법은 대회 안내를 확인해 주세요.

`test.py`는 입력 CSV에 있는 행만 가지고 예측해야 합니다. 운영진이 코드를 확인할 때 5번과 같은 방식으로 Test를 나누어 다시 돌려봅니다.

*참가자분들의 독창적인 아이디어로 다양한 전처리 기법과 모델링을 통해 열폭주 예측 성능을 향상시켜보시기 바랍니다. 응원하겠습니다!*